In [1]:
import sys

assert sys.version_info >= (3,10)

In [2]:
from packaging.version import Version
import torch

assert Version(torch.__version__) >= Version("2.6.0")

In [3]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

'cuda'

In [4]:
import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('legend', fontsize=14)
plt.rc("xtick",labelsize=10)
plt.rc("ytick", labelsize=10)
plt.rc("axes",labelsize=14, titlesize=14)

Vision Transformers

ViT From scratch

In [5]:
import torch
import torch.nn as nn 

class PatchEmbedding(nn.Module):
    def __init__(self, in_channels, embed_dim, patch_size=16):
        super().__init__()
        self.conv2d = nn.Conv2d(in_channels, embed_dim,
                                kernel_size=patch_size, stride=patch_size)

    def forward(self, X):
        X = self.conv2d(X)
        X = X.flatten(start_dim=2)
        return X.transpose(1,2)

In [6]:
class ViT(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3,
                 num_classes=1000, embed_dim=768, depth=12, num_heads=12,
                 ff_dim=3072, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(in_channels, embed_dim,)
        cls_init = torch.randn(1, 1, embed_dim) * 0.02
        self.cls_token = nn.Parameter(cls_init)
        num_patches = (img_size // patch_size) **2
        pos_init = torch.randn(1, num_patches + 1, embed_dim) * 0.02
        self.pos_embed = nn.Parameter(pos_init)
        self.dropout = nn.Dropout(p=dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=ff_dim,
            dropout=dropout, activation="gelu", batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.layer_norm = nn.LayerNorm(embed_dim)
        self.output = nn.Linear(embed_dim, num_classes)

    def forward(self, X):
        Z  = self.patch_embed(X)
        cls_expd = self.cls_token.expand(Z.shape[0], -1, -1)
        Z = torch.cat((cls_expd, Z), dim=1)
        Z = Z + self.pos_embed
        Z = self.dropout(Z)
        Z = self.encoder(Z)
        Z = self.layer_norm(Z[:, 0])
        logits = self.output(Z)
        return logits

In [7]:
vit_model = ViT(
    img_size=224, patch_size=16, in_channels=3, num_classes=1000, 
    embed_dim=768, depth=12, num_heads=12, ff_dim=372, dropout=0.1
)
batch = torch.randn(4, 3,224, 224)
logits = vit_model(batch)

In [13]:
logits.shape

torch.Size([4, 1000])

Fine_Tuning a Pretrained ViT

In [ ]:
from datasets import load_dataset

pets = load_dataset("timm/oxford-iiit-pet")

In [ ]:
num_rows, num_cols = 2, 5
plt.figure(figsize=(num_cols * 2.5, num_rows * 2))
class_names = pets["train"].features["label"].names
for i in range(num_rows * num_cols):
    plt.subplot(num_rows, num_cols, i + 1)
    example = pets["train"][i]
    plt.imshow(example["image"])
    plt.title(class_names[example["label"]])
    plt.axis("off")


In [ ]:
from transformers import ViTForImageClassification, AutoImageProcessor

model_id = "google/vit-base-patch16-224-in21k"
vit_model = ViTForImageClassification.from_pretrained(model_id, num_labels=37)
vit_processor = AutoImageProcessor.from_pretrained(model_id, use_fast=True)

In [ ]:
num_rows, num_cols = 2, 5
plt.figure(figsize=(num_cols * 2.5, num_rows *2))
class_names = pets["train"].features["label"].names
for i in range(num_rows * num_cols):
    plt.subplot(num_rows, num_cols, i+1)
    example = pets["train"][i]
    preprocessed = vit_processor(example["image"])["pixel_values"][0]
    plt.title((preprocessed.permute(1, 2, 0) + 1.) / 2.)
    plt.axis("off")

In [ ]:
def vit_collate_fn(batch):
    images = [example["image"] for example in batch]
    labels = [example["label"] for example in batch]
    inputs = vit_processor(images, return_tensors="pt", do_convert_rgb=True)
    inputs["labels"] = torch.tensro(labels)
    return inputs

In [ ]:
def compute_accuracy(logits_and_labels):
    logits, labels = logits_and_labels
    preds = torch.tensor(logits).argmax(dim=1)
    labels = torch.tensor(labels)
    accuracy = (preds == labels).float().mean()
    return {"accuracy": accuracy.item()}

In [ ]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments("my_pets_vit", per_device_train_batch_size=16,
                         eval_strategy="epoch", num_train_epochs=3,
                         remove_unused_columns=False,
                         report_to="none")

trainer = Trainer(model=vit_model, args=args,data_collator=vit_collate_fn,
                  train_dataset=pets["train"], eval_dataset=pets["test"],
                  compute_metrics=compute_accuracy)
train_output = trainer.train()

Let's try fine_tuning a DeiT model instead:

Fine-tuning a pretrained DeiT

In [ ]:
from transformers import DeiTForImageClassification, AutoImageProcessor

model_id = "facebook/deit-base-distilled-patch16-224"
deit_model = DeiTForImageClassification.from_pretrained(model_id, num_labels=37)
deit_processor = AutoImageProcessor.from_pretrained(model_id, use_fast=True)

In [ ]:
def deit_collate_fn(batch):
    images = [example["image"] for example in batch]
    labels = [example["label"] for example in batch]
    inputs = deit_processor(images, return_tensors="pt", do_convert_rgb=True)
    inputs["labels"] = torch.tensor(labels)
    return inputs

In [ ]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments("my_pets_deit", per_device_train_batch_size=16,
                         eval_strategy="epoch", num_train_epochs=3,
                         remove_unused_columns=False,
                         report_to="none")
trainer = Trainer(model=deit_model, args=args, data_collator=deit_collate_fn,
                  train_dataset=pets["train"], eval_dataset =pets["test"],
                  compute_metrics=compute_accuracy)
train_output = trainer.train()

Unsupervised image segmentation usin DINO

In [ ]:
from PIL import Image
import urllib.request

image_url = 'http://images.cocodataset.org/val2017/000000039769.jpg'
image = Image.open(urllib.request.urlopen(image_url))
image

In [ ]:
from transformers import AutoImageProcessor, AutoModel

model_id = "facebook/dino-vitb8"
model  = AutoModel.from_pretrained(model_id, output_attentions=True)
processor = AutoImageProcessor.from_pretrained(model_id, do_convert_rgb=True)

In [ ]:
inputs  = processor(images=image, return_tensors="pt")
with torch.no_grad():
    output = model(**inputs)

cls_token_output = output.last_hidden_state[:, 0]
cls_token_output.shape

In [ ]:
last_layer_attention_maps = output.attentions[-1]
cls_attn = last_layer_attention_maps[0, :, 1:,0]
cls_attn.shape

In [ ]:
import torchvision.transforms.functional as TF

num_heads, num_patches = cls_attn.shape
size = int(num_patches ** 0.5)
plt.figure(figsize = (12, 7))
for head_index in range(12):
    plt.subplot(3, 4, head_index + 1)
    attn = cls_attn[head_index].reshape(size, size)
    attn_map = attn.unsqueeze(0).unsqueze(0)
    attn_resized = TF.resize(attn_map, image.size[::-1],
                             interpolation=TF.InterpolationMode.BILINEAR)[0,0]
    plt.imshow(image)
    plt.imshow(attn_resized.numpy(), cmap='jet', alpha=o.5)
    plt.axis('off')

plt.show()

Multimodal Transformers

CLIP

In [ ]:
from transformers import pipeline

model_id = "openai/clip-vit-base-patch32"
clip_pipeline = pipeline(task="zero-shot-image-classification", model=model_id,
device_map="auto", dtype="auto")
candidate_labels = ["cricket", "ladybug", "spider"]
image_url = "https://homl.info/ladybug"
results  = clip_pipeline(image_url, candidate_labels=candidate_labels,
                         hypothesis_template="This is a photo of a {}.")


Correct! now let's build a flower classifier:

In [ ]:
candidate_labels2 = ["dandelion", "lily", "popy", "rose", "sunflower"]
results2 = clip_pipeline(image_url, candidate_labels=candidate_labels2,
                         hypothesis_template="This is a photo of a {}.")
results2

In [ ]:
import PIL
import urllib.request
from transformers import CLIPProcessor, CLIPModel

clip_processor = CLIPProcessor.from_pretrained(model_id)
clip_model = CLIPModel.from_pretrained(model_id)
image = PIL.Image.open(urllib.request.urlopen(image_url)).conver("RGB")
captions  = [f'This is a photo of a {label}.' for label in candidate_labels]
inputs = clip_processor(text=captions, images=[image], return_tensors="pt",
                        padding=True)
with torch.no_grad():
    outputs = clip_model(**inputs)

text_features = outputs.text_embeds
image_features = outputs.image_embeds


If you want to encode images and text separately, you can use the following cod:

In [ ]:
from transformers import CLIPTokenizer, CLIPImageProcessor, CLIPModel
from PIL import Image
import urllib.request

model_id = "openai/clip-vit-base-patch32"
clip_tokenizer = CLIPTokenizer.from_pretrained(model_id)
clip_image_processor = CLIPImageProcessor.from_pretrained(model_id)
clip_model = CLIPModel.from_pretrained(model_id)

image  = Image.open(urllib.request.urlopen(image_url)).convert("RGB")
image_inputs = clip_image_processor(images=image, return_tensors="pt")
with torch.no_grad():
    image_features =clip_model.get_image_features(**image_inputs)
    image_features /= image_features.norm(dim=1, keepdim=True)

captions = [f"This is a photo of a {label}." for label in candidate_labels]
text_inputs = clip_tokenizer(captions, padding=True, return_tensors="pt")
with torch.no_grad():
    text_features = clip_model.get_text_features(**text_inputs)
    text_features /= text_features.norm(dim=1, keepdim=True)

We get the same probabilities, of course:

In [ ]:
similarities = image_features @ text_features.T 
temperature = clip_model.logit_scale.detach().exp()
rescaled_similarities = similarities * temperature
probabilities = torch.nn.functional.softmax(rescaled_similarities, dim=1)
probabilities

Perceiver"


In [ ]:
class FourierPositionalEncoding(nn.Module):
    def __init__(self, num_bands, max_resolution):
        super().__init__()
        self.num_bands = num_bands
        self.max_resolution = max_resolution
        frequencies = torch.linspace(1.0, max_resolution/ 2, steps=num_bands)
        self.register_buffer("frequencies", frequencies)

    def forward(self, X):
        out = [X]
        for freq in self.frequencies:
            angles = torch.pi * freq *X
            out += ['angles.sin(), angles.cos()']
        return torch.cat(out, dim=1)

In [ ]:
H, W = 224, 224
num_bands = 6
coords_y = torch.linspace(-1, 1, H)
coords_x = torch.linspace(-1, 1, W)
grid_y, grid_x = torch.meshgrid(coords_y, coords_x, indexing="ij")
pos  = torch.stack([grid_x, grid_y], dim=-1)

X  = torch.rand(10, 224 * 224, 3)
fourier_pos_enc = FourierPositionalEncoding(num_bands=num_bands, max_resolution=H)
pos_encodings = fourier_pos_enc(pos)